In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/llama70b-results-jsonl/llama70b_results.jsonl
/kaggle/input/llma-variation-val/llama70b_results_val.jsonl


In [2]:
import sys
import json
import ast
import re
import torch
import gc
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from tqdm import tqdm
from peft import PeftModel

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ID = 'vohuutridung/mt5-large-absa'
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map='auto',
)

eval_model.eval()

2025-12-31 13:11:06.775099: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767186666.957196      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767186667.013188      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767186667.456149      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767186667.456186      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767186667.456189      55 computation_placer.cc:177] computation placer alr

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/784 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 1024)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 1024)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=Fals

In [3]:
GEN_CACHE = {}

In [4]:
# --- 2. CÁC HÀM HỖ TRỢ ---
CATEGORIES = {
    "TỔNG_QUAN","PIN","HIỆU_NĂNG","MÁY_ẢNH","MÀN_HÌNH",
    "GIÁ_CẢ","TÍNH_NĂNG","THIẾT_KẾ","DỊCH_VỤ&PHỤ_KIỆN","LƯU TRỮ"
}
SENTIMENT = {"TÍCH_CỰC","TIÊU_CỰC","TRUNG_LẬP"}
QUAD_RE = re.compile(
    r"\[\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*,\s*'([^']*)'\s*\]"
)

def extract_json_safe(raw_text):
    """
    Robust parser for ViT5 / seq2seq ABSA output.
    """
    if not raw_text:
        return []

    s = str(raw_text)

    results = []
    for a, c, se, o in QUAD_RE.findall(s):
        a, c, se, o = a.strip(), c.strip(), se.strip(), o.strip()

        # Hard validation (quan trọng)
        if c not in CATEGORIES:
            continue
        if se not in SENTIMENT:
            continue

        results.append([a, c, se, o])

    return results

@torch.no_grad()
def get_absa_prediction_debug(reviews):
    results = [None] * len(reviews)
    uncached_reviews = []
    uncached_indices = []

    # --- 1. Check cache ---
    for i, review in enumerate(reviews):
        if review in GEN_CACHE:
            results[i] = GEN_CACHE[review]
        else:
            uncached_reviews.append(review)
            uncached_indices.append(i)

    if len(uncached_reviews) == 0:
        return results

    # --- 2. Tokenize (Seq2Seq không cần system prompt) ---
    inputs = eval_tokenizer(
        uncached_reviews,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(device)

    # --- 3. Batch generate ---
    outputs = eval_model.generate(
        **inputs,
        max_new_tokens=256,
    )

    decoded = eval_tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # --- 4. Parse + map lại đúng vị trí ---
    for i, out_idx in enumerate(uncached_indices):
        clean_text = decoded[i].strip()
        parsed = extract_json_safe(clean_text)

        results[out_idx] = (parsed, clean_text)
        GEN_CACHE[uncached_reviews[i]] = (parsed, clean_text)

    return results


def calculate_f1(pred, gold):
    if not pred and not gold: return 1.0
    if not pred or not gold: return 0.0
    def clean(s): return str(s).strip().lower().replace("_", " ").replace("-", " ")
    try:
        pred_set = set([tuple(clean(item) for item in x) for x in pred if isinstance(x, list) and len(x) == 4])
        gold_set = set([tuple(clean(item) for item in x) for x in gold if isinstance(x, list) and len(x) == 4])
        tp = len(pred_set & gold_set)
        if tp == 0: return 0.0
        p = tp / len(pred_set); r = tp / len(gold_set)
        return 2 * (p * r) / (p + r)
    except Exception as e: 
        print("F1 error:", e)
        return 0.0


# --- 3. MAIN PROCESS VỚI LOGIC P+ ---
def process_full_logic(input_file, output_file, sample_size=None, GLOBAL_BATCH=16):
    results = []
    count_original_chosen = 0
    parse_miss = 0

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if sample_size: lines = lines[:sample_size]
    except Exception as e:
        print(f"❌ Lỗi file: {e}"); 
        return

    print(f"🚀 Bắt đầu xử lý {len(lines)} mẫu | GLOBAL_BATCH={GLOBAL_BATCH}")

    for batch_start in tqdm(range(0, len(lines), GLOBAL_BATCH)):
        batch_lines = lines[batch_start: batch_start + GLOBAL_BATCH]
        all_texts = []
        metas = []  # (data, gold, start_idx, n_texts)

        # Gom batch
        for line in batch_lines:
            data = json.loads(line)
            gold = ast.literal_eval(data['gold_label'])
    
            original = data["original"]
            variations = data["variations"]
    
            texts = [original] + variations
            start_idx = len(all_texts)
    
            all_texts.extend(texts)
            metas.append((data, gold, original, variations, start_idx, len(texts)))

        preds = get_absa_prediction_debug(all_texts)

        for data, gold, original, variations, start_idx, n_texts in metas:
            sample_preds = preds[start_idx: start_idx + n_texts]
            
            (pred_s, raw_s) = sample_preds[0]
            if len(pred_s) == 0: parse_miss += 1
            f1_s = calculate_f1(pred_s, gold)
            
            var_preds = sample_preds[1:]
            f1_vars = []
            for pred_v, _ in var_preds:
                if len(pred_v) == 0: parse_miss += 1
                f1_vars.append(calculate_f1(pred_v, gold))

            # --- BƯỚC 3: LOGIC CHỌN P+ ---
            p_plus = []
            
            # CASE A: CÂU ĐƠN -> Chọn biến thể có số câu = 1 (khớp với số quadruplet)
            if len(gold) == 1: 
                found_short = False
                for i, v in enumerate(data['variations']):
                    # [SỬA QUAN TRỌNG]: Lọc bỏ chuỗi rỗng để đếm đúng số câu
                    parts = [s for s in re.split(r'[.!?]+', v.strip()) if s.strip()]
                    
                    if len(parts) == 1: # matches the number of quadruplets [cite: 255]
                        p_plus.append(v)
                        found_short = True
                
                if not found_short:
                    # Fallback: Nếu không tìm thấy biến thể nào phù hợp, giữ Original
                    p_plus = [data['original']]
                    count_original_chosen += 1

            # CASE B: CÂU PHỨC -> Chọn dựa trên so sánh F1
            else:
                higher_f1_vars = []
                for v, f1_v in zip(data['variations'], f1_vars):
                    if f1_v > f1_s:
                        higher_f1_vars.append(v)
                
                if higher_f1_vars:
                    # select all distinct s' with higher score [cite: 257]
                    p_plus = list(set(higher_f1_vars))
                elif any(f1_v == f1_s for f1_v in f1_vars):
                    # if scores are equal, s is retained [cite: 257]
                    p_plus = [data['original']]
                    count_original_chosen += 1
                else:
                    # if s has higher score, no preferred sentence is chosen 
                    p_plus = [data['original']]
                    count_original_chosen += 1

            # Lưu kết quả
            data['p_plus'] = p_plus
            data['f1_original'] = f1_s
            data['f1_variations'] = f1_vars
            results.append(data)

        if batch_start % 10 == 0 and batch_start != 0: print(f"Parse miss: {parse_miss}")
                

    # Write output
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in results:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    
    print(f"\n💾 Đã lưu file: {output_file}")
    print(f"{'='*40}")
    print(f"📊 TỔNG KẾT:")
    print(f"   - Tổng số mẫu: {len(results)}")
    print(f"   - Số lần chọn P+ là Original: {count_original_chosen}")
    if len(results) > 0:
        print(f"   - Tỷ lệ giữ nguyên: {(count_original_chosen/len(results))*100:.2f}%")
    else:
        print(f"   - Tỷ lệ giữ nguyên: 0%")
    print(f"     - Parse miss: {parse_miss}")
    print(f"{'='*40}")

In [5]:
# --- EXECUTION ---
INPUT_PATH = '/kaggle/input/llama70b-results-jsonl/llama70b_results.jsonl'
OUTPUT_PATH = 'pp_train.jsonl'
process_full_logic(INPUT_PATH, OUTPUT_PATH, sample_size=None)

🚀 Bắt đầu xử lý 9999 mẫu | GLOBAL_BATCH=16


  1%|          | 6/625 [01:10<2:02:13, 11.85s/it]

Parse miss: 0


  2%|▏         | 11/625 [02:14<2:09:19, 12.64s/it]

Parse miss: 0


...

Parse miss: 4


 99%|█████████▉| 621/625 [2:06:24<00:50, 12.66s/it]

Parse miss: 4


100%|██████████| 625/625 [2:07:13<00:00, 12.21s/it]



💾 Đã lưu file: pp_train.jsonl
📊 TỔNG KẾT:
   - Tổng số mẫu: 9999
   - Số lần chọn P+ là Original: 6887
   - Tỷ lệ giữ nguyên: 68.88%
     - Parse miss: 4


In [6]:
# --- EXECUTION ---
INPUT_PATH = '/kaggle/input/llma-variation-val/llama70b_results_val.jsonl'
OUTPUT_PATH = 'pp_val.jsonl'
process_full_logic(INPUT_PATH, OUTPUT_PATH, sample_size=None)

🚀 Bắt đầu xử lý 300 mẫu | GLOBAL_BATCH=16


 32%|███▏      | 6/19 [01:19<02:55, 13.47s/it]

Parse miss: 0


 58%|█████▊    | 11/19 [02:22<01:44, 13.06s/it]

Parse miss: 0


 84%|████████▍ | 16/19 [03:26<00:38, 12.83s/it]

Parse miss: 0


100%|██████████| 19/19 [04:01<00:00, 12.69s/it]


💾 Đã lưu file: pp_val.jsonl
📊 TỔNG KẾT:
   - Tổng số mẫu: 300
   - Số lần chọn P+ là Original: 206
   - Tỷ lệ giữ nguyên: 68.67%
     - Parse miss: 0
